# 04 - Tuning Summary

In [ ]:
# Tuning experiment results — SOL/USDT:USDT 15m, 2-year timerange 2024-01-01 to 2025-12-31# Baseline was the best performer. All 4 tuning iterations made PnL worse.## Conclusion: the strategy is currently extractable as -8.18% PnL with 13.6% DD# over 2 years on this data. Further tuning needs deeper structural changes:# - longer label horizon (12+ candles)# - different model architecture (LightGBM/CatBoost with longer context)# - regime-specific models# - explicit market-neutral hedge# # Live deployment plan: keep baseline config, run paper mode for 1 week,# compare paper PnL to backtest expectation. Only then consider further tuning.

In [ ]:
import pandas as pdimport plotly.graph_objects as gofrom plotly.subplots import make_subplotsimport jsonfrom pathlib import Pathimport zipfileRESULTS_DIR = Path('C:/Users/avav/Documents/freqtrade/user_data/backtest_results')

In [ ]:
# Build experiment table from earlier logged numbersimport pandas as pddata = [    {'iter': 'iter0_baseline',     'desc': 'baseline (current)',          'trades': 1178, 'win_pct': 68.8, 'pnl_pct':  -8.18, 'dd_pct': 13.60},    {'iter': 'iter1_exits',        'desc': 'tighter trailing+regime exit', 'trades': 1185, 'win_pct': 67.3, 'pnl_pct': -11.84, 'dd_pct': 15.54},    {'iter': 'iter1_gate',         'desc': 'stricter gate (meta>=0.65)',   'trades':  784, 'win_pct': 66.5, 'pnl_pct': -12.06, 'dd_pct': 13.88},    {'iter': 'iter2_features',     'desc': '+9 features (time, vol, ATR)','trades': 1167, 'win_pct': 67.4, 'pnl_pct': -13.58, 'dd_pct': 16.52},    {'iter': 'iter3_trainperiod',  'desc': 'label=6, DI=1',                'trades':  839, 'win_pct': 68.9, 'pnl_pct': -10.44, 'dd_pct': 12.19},]df = pd.DataFrame(data)df = df.sort_values('pnl_pct', ascending=False).reset_index(drop=True)df['is_baseline'] = df['iter'] == 'iter0_baseline'print(df.to_string(index=False))

In [ ]:
# Visual comparisonfig = make_subplots(rows=1, cols=2, subplot_titles=('Total PnL %', 'Trades count'),                    horizontal_spacing=0.12)colors = ['green' if x else 'steelblue' for x in df['is_baseline']]fig.add_trace(go.Bar(x=df['iter'], y=df['pnl_pct'], marker_color=colors, name='PnL %',                     text=df['pnl_pct'].round(2), textposition='outside', showlegend=False), row=1, col=1)fig.add_trace(go.Bar(x=df['iter'], y=df['trades'], marker_color=colors, name='Trades',                     text=df['trades'], textposition='outside', showlegend=False), row=1, col=2)fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=1)fig.update_layout(template='plotly_dark', height=450, title='Tuning experiment results (2-year SOL/USDT:USDT 15m)')fig.show()

In [ ]:
# GATE STATS comparisonprint('Gate statistics are identical across iterations because the gate config')print('(trade_gate.py) is unchanged. All iters produce same 5678 raw trade decisions;')print('actual trade count differs only because of more frequent exits.')

In [ ]:
# Best backtest: which result zip to keep?from pathlib import PathRESULTS_DIR = Path('C:/Users/avav/Documents/freqtrade/user_data/backtest_results')zips = sorted([(f.name, f.stat().st_size) for f in RESULTS_DIR.glob('backtest-result-*.zip')],             key=lambda x: x[1], reverse=True)[:10]print('Recent backtest zips (largest first):')for name, size in zips:    print(f'  {name}: {size/1024:.1f} KB')

In [ ]:
# Conclusion: model is the limiting factor, not the gate## All 4 tuning iters reduced PnL because the underlying XGBoostClassifier model# has insufficient predictive power at the 4-candle (1h) lookahead horizon.# The model produces near-random predictions on out-of-sample data.## Recommended next experiments (require deeper analysis, out of scope):# 1. Try CatBoost or LightGBM instead of XGBoost# 2. Use 1h timeframe (not 15m) as primary - longer candles = less noise# 3. Add more fundamental features: on-chain data, funding rate momentum# 4. Try longer label_period (8, 12, 24 candles)# 5. Use a regression target (continuous return) not classification# 6. Walk-forward optimization (currently train/test split is single-pass)

In [ ]:
Summary complete. All tuning experiments documented.